In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

In [ ]:
import zipfile

# Define the path to your zip file
# IMPORTANT: Replace '/content/gdrive/MyDrive/path/to/your_file.zip' with the actual path to your zip file.
zip_file_path = '/content/gdrive/MyDrive/Dataset.zip' # Placeholder path

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        print(f"Contents of {zip_file_path}:")
        for file_name in zip_ref.namelist():
            print(file_name)
except FileNotFoundError:
    print(f"Error: The file '{zip_file_path}' was not found. Please ensure the path is correct and the file exists.")
except zipfile.BadZipFile:
    print(f"Error: '{zip_file_path}' is not a valid zip file or is corrupted.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
import zipfile
import os

# The zip_file_path is already defined from the previous step
# zip_file_path = '/content/gdrive/MyDrive/Dataset.zip'

# Define the directory where you want to extract the files
extract_dir = '/content/extracted_dataset'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        print(f"Extracting all contents from '{zip_file_path}' to '{extract_dir}'...")
        zip_ref.extractall(extract_dir)
        print("Extraction complete!")
except FileNotFoundError:
    print(f"Error: The file '{zip_file_path}' was not found. Please ensure the path is correct and the file exists.")
except zipfile.BadZipFile:
    print(f"Error: '{zip_file_path}' is not a valid zip file or is corrupted.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [ ]:
!pip install ultralytics

In [ ]:
import shutil
import os

# Define source (read-only) and destination (writable)
source_dir = '/kaggle/input/datasets/anonynous/drone-detact-dataset/Dataset'
working_dir = '/kaggle/working/Dataset'

# Copy the dataset to the working directory
if not os.path.exists(working_dir):
    shutil.copytree(source_dir, working_dir)
    print("Dataset copied to writable directory!")
else:
    print("Dataset already exists in working directory.")

In [ ]:
import os

# Define the path where the new yaml file will be saved
yaml_file_path = '/kaggle/working/dataset.yaml'

# Define the updated YAML content
yaml_content = """# YOLO Dataset Configuration for Robust Lightweight Tiny UAV Detection
# Dataset: Drone vs Bird Detection

path: /kaggle/working/Dataset # UPDATED: writable dataset root dir
train: train/images  # train images (relative to 'path')
val: valid/images  # val images (relative to 'path')
test: test/images  # test images (optional, relative to 'path')

# Classes
names:
  0: drone  # UAV/Drone class

# Number of classes
nc: 1  # Single class: drone detection
"""

# Write the content to the file
with open(yaml_file_path, 'w') as f:
    f.write(yaml_content)

print(f"Updated dataset.yaml successfully created at: {yaml_file_path}")

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from matplotlib.gridspec import GridSpec
from ultralytics import YOLO

def create_comprehensive_graphs(results_dir, model, dataset_yaml, train_metrics=None, val_metrics=None, test_metrics=None):
    """
    Create comprehensive training, validation, and test graphs for presentation.

    Args:
        results_dir: Path to results directory
        model: Trained YOLO model
        dataset_yaml: Path to dataset configuration
    """
    print("\n" + "="*50)
    print("Generating comprehensive graphs...")
    print("="*50)

    # Set style for professional presentation
    try:
        plt.style.use('seaborn-v0_8-darkgrid')
    except:
        try:
            plt.style.use('seaborn-darkgrid')
        except:
            plt.style.use('dark_background')
    sns.set_palette("husl")

    # Read training results CSV
    results_csv = Path(results_dir) / "results.csv"
    if not results_csv.exists():
        print(f"Warning: results.csv not found at {results_csv}")
        print("Graphs will be generated from validation metrics only.")
        return

    # Load training history
    df = pd.read_csv(results_csv)

    # Create output directory for graphs
    graphs_dir = Path(results_dir) / "comprehensive_graphs"
    graphs_dir.mkdir(exist_ok=True)

    # ========== 1. COMPREHENSIVE TRAINING DASHBOARD ==========
    print("\n1. Creating comprehensive training dashboard...")
    fig = plt.figure(figsize=(20, 12))
    gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

    epochs = df.index + 1

    # 1.1 Training & Validation Loss
    ax1 = fig.add_subplot(gs[0, 0])
    if 'train/box_loss' in df.columns:
        ax1.plot(epochs, df['train/box_loss'], label='Train Box Loss', linewidth=2, alpha=0.8)
    if 'train/cls_loss' in df.columns:
        ax1.plot(epochs, df['train/cls_loss'], label='Train Class Loss', linewidth=2, alpha=0.8)
    if 'train/dfl_loss' in df.columns:
        ax1.plot(epochs, df['train/dfl_loss'], label='Train DFL Loss', linewidth=2, alpha=0.8)
    if 'val/box_loss' in df.columns:
        ax1.plot(epochs, df['val/box_loss'], label='Val Box Loss', linewidth=2, linestyle='--', alpha=0.8)
    if 'val/cls_loss' in df.columns:
        ax1.plot(epochs, df['val/cls_loss'], label='Val Class Loss', linewidth=2, linestyle='--', alpha=0.8)
    if 'val/dfl_loss' in df.columns:
        ax1.plot(epochs, df['val/dfl_loss'], label='Val DFL Loss', linewidth=2, linestyle='--', alpha=0.8)
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(loc='best', fontsize=9)
    ax1.grid(True, alpha=0.3)

    # 1.2 mAP Curves
    ax2 = fig.add_subplot(gs[0, 1])
    if 'metrics/mAP50(B)' in df.columns:
        ax2.plot(epochs, df['metrics/mAP50(B)'], label='mAP50', linewidth=2.5, color='#2ecc71')
    if 'metrics/mAP50-95(B)' in df.columns:
        ax2.plot(epochs, df['metrics/mAP50-95(B)'], label='mAP50-95', linewidth=2.5, color='#e74c3c')
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('mAP', fontsize=12, fontweight='bold')
    ax2.set_title('Mean Average Precision (mAP)', fontsize=14, fontweight='bold')
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, alpha=0.3)

    # 1.3 Precision & Recall
    ax3 = fig.add_subplot(gs[0, 2])
    if 'metrics/precision(B)' in df.columns:
        ax3.plot(epochs, df['metrics/precision(B)'], label='Precision', linewidth=2.5, color='#3498db')
    if 'metrics/recall(B)' in df.columns:
        ax3.plot(epochs, df['metrics/recall(B)'], label='Recall', linewidth=2.5, color='#f39c12')
    ax3.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax3.set_title('Precision & Recall', fontsize=14, fontweight='bold')
    ax3.legend(loc='best', fontsize=10)
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim([0, 1])

    # 1.4 Learning Rate
    ax4 = fig.add_subplot(gs[1, 0])
    if 'lr/pg0' in df.columns:
        ax4.plot(epochs, df['lr/pg0'], label='Learning Rate', linewidth=2, color='#9b59b6')
    ax4.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
    ax4.set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    ax4.legend(loc='best', fontsize=10)
    ax4.grid(True, alpha=0.3)
    ax4.set_yscale('log')

    # 1.5 Combined Loss
    ax5 = fig.add_subplot(gs[1, 1])
    if 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
        train_total = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
        val_total = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
        ax5.plot(epochs, train_total, label='Train Total Loss', linewidth=2.5, color='#e74c3c')
        ax5.plot(epochs, val_total, label='Val Total Loss', linewidth=2.5, color='#3498db', linestyle='--')
    ax5.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax5.set_ylabel('Total Loss', fontsize=12, fontweight='bold')
    ax5.set_title('Total Training & Validation Loss', fontsize=14, fontweight='bold')
    ax5.legend(loc='best', fontsize=10)
    ax5.grid(True, alpha=0.3)

    # 1.6 Loss Components Comparison
    ax6 = fig.add_subplot(gs[1, 2])
    if 'val/box_loss' in df.columns and 'val/cls_loss' in df.columns and 'val/dfl_loss' in df.columns:
        box_loss_final = df['val/box_loss'].iloc[-10:].mean()
        cls_loss_final = df['val/cls_loss'].iloc[-10:].mean()
        dfl_loss_final = df['val/dfl_loss'].iloc[-10:].mean()
        components = ['Box Loss', 'Class Loss', 'DFL Loss']
        values = [box_loss_final, cls_loss_final, dfl_loss_final]
        colors = ['#e74c3c', '#3498db', '#2ecc71']
        bars = ax6.bar(components, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
        ax6.set_ylabel('Average Loss (Last 10 Epochs)', fontsize=12, fontweight='bold')
        ax6.set_title('Validation Loss Components', fontsize=14, fontweight='bold')
        ax6.grid(True, alpha=0.3, axis='y')
        # Add value labels on bars
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax6.text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # 1.7 Metrics Summary Table
    ax7 = fig.add_subplot(gs[2, :])
    ax7.axis('off')

    # Get final metrics
    metrics_data = []
    if 'metrics/mAP50(B)' in df.columns:
        metrics_data.append(['mAP50', f"{df['metrics/mAP50(B)'].iloc[-1]:.4f}",
                           f"{df['metrics/mAP50(B)'].max():.4f}"])
    if 'metrics/mAP50-95(B)' in df.columns:
        metrics_data.append(['mAP50-95', f"{df['metrics/mAP50-95(B)'].iloc[-1]:.4f}",
                           f"{df['metrics/mAP50-95(B)'].max():.4f}"])
    if 'metrics/precision(B)' in df.columns:
        metrics_data.append(['Precision', f"{df['metrics/precision(B)'].iloc[-1]:.4f}",
                           f"{df['metrics/precision(B)'].max():.4f}"])
    if 'metrics/recall(B)' in df.columns:
        metrics_data.append(['Recall', f"{df['metrics/recall(B)'].iloc[-1]:.4f}",
                           f"{df['metrics/recall(B)'].max():.4f}"])

    if metrics_data:
        table = ax7.table(cellText=metrics_data,
                         colLabels=['Metric', 'Final Value', 'Best Value'],
                         cellLoc='center',
                         loc='center',
                         bbox=[0, 0, 1, 1])
        table.auto_set_font_size(False)
        table.set_fontsize(12)
        table.scale(1, 2)
        for i in range(len(metrics_data) + 1):
            for j in range(3):
                cell = table[(i, j)]
                if i == 0:  # Header
                    cell.set_facecolor('#34495e')
                    cell.set_text_props(weight='bold', color='white')
                else:
                    cell.set_facecolor('#ecf0f1' if i % 2 == 0 else 'white')
        ax7.set_title('Training Metrics Summary', fontsize=16, fontweight='bold', pad=20)

    plt.suptitle('Comprehensive Training Dashboard - UAV Detection Model',
                 fontsize=18, fontweight='bold', y=0.995)
    plt.savefig(graphs_dir / '01_comprehensive_dashboard.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: comprehensive_dashboard.png")

    # ========== 2. DETAILED LOSS ANALYSIS ==========
    print("2. Creating detailed loss analysis...")
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Detailed Loss Analysis - Training vs Validation', fontsize=16, fontweight='bold')

    # 2.1 Box Loss
    ax = axes[0, 0]
    if 'train/box_loss' in df.columns:
        ax.plot(epochs, df['train/box_loss'], label='Train', linewidth=2, alpha=0.8)
    if 'val/box_loss' in df.columns:
        ax.plot(epochs, df['val/box_loss'], label='Validation', linewidth=2, linestyle='--', alpha=0.8)
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Box Loss', fontsize=11, fontweight='bold')
    ax.set_title('Box Loss', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # 2.2 Class Loss
    ax = axes[0, 1]
    if 'train/cls_loss' in df.columns:
        ax.plot(epochs, df['train/cls_loss'], label='Train', linewidth=2, alpha=0.8)
    if 'val/cls_loss' in df.columns:
        ax.plot(epochs, df['val/cls_loss'], label='Validation', linewidth=2, linestyle='--', alpha=0.8)
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Class Loss', fontsize=11, fontweight='bold')
    ax.set_title('Class Loss', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # 2.3 DFL Loss
    ax = axes[1, 0]
    if 'train/dfl_loss' in df.columns:
        ax.plot(epochs, df['train/dfl_loss'], label='Train', linewidth=2, alpha=0.8)
    if 'val/dfl_loss' in df.columns:
        ax.plot(epochs, df['val/dfl_loss'], label='Validation', linewidth=2, linestyle='--', alpha=0.8)
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('DFL Loss', fontsize=11, fontweight='bold')
    ax.set_title('DFL Loss', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    # 2.4 Total Loss Comparison
    ax = axes[1, 1]
    if all(col in df.columns for col in ['train/box_loss', 'train/cls_loss', 'train/dfl_loss']):
        train_total = df['train/box_loss'] + df['train/cls_loss'] + df['train/dfl_loss']
        val_total = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
        ax.plot(epochs, train_total, label='Train Total', linewidth=2.5, color='#e74c3c')
        ax.plot(epochs, val_total, label='Validation Total', linewidth=2.5, color='#3498db', linestyle='--')
        # Add best epoch marker
        best_epoch = val_total.idxmin() + 1
        best_loss = val_total.min()
        ax.plot(best_epoch, best_loss, 'ro', markersize=12, label=f'Best: Epoch {best_epoch}')
        ax.annotate(f'Epoch {best_epoch}\nLoss: {best_loss:.4f}',
                   xy=(best_epoch, best_loss), xytext=(10, 10),
                   textcoords='offset points', fontsize=10, fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Total Loss', fontsize=11, fontweight='bold')
    ax.set_title('Total Loss Comparison', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(graphs_dir / '02_detailed_loss_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: detailed_loss_analysis.png")

    # ========== 3. METRICS EVOLUTION ==========
    print("3. Creating metrics evolution graphs...")
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Model Performance Metrics Evolution', fontsize=16, fontweight='bold')

    # 3.1 mAP50
    ax = axes[0, 0]
    if 'metrics/mAP50(B)' in df.columns:
        ax.plot(epochs, df['metrics/mAP50(B)'], linewidth=2.5, color='#2ecc71', label='mAP50')
        best_epoch = df['metrics/mAP50(B)'].idxmax() + 1
        best_map = df['metrics/mAP50(B)'].max()
        ax.plot(best_epoch, best_map, 'ro', markersize=12)
        ax.axhline(y=best_map, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_map:.4f}')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('mAP50', fontsize=11, fontweight='bold')
    ax.set_title('mAP50 Evolution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    # 3.2 mAP50-95
    ax = axes[0, 1]
    if 'metrics/mAP50-95(B)' in df.columns:
        ax.plot(epochs, df['metrics/mAP50-95(B)'], linewidth=2.5, color='#e74c3c', label='mAP50-95')
        best_epoch = df['metrics/mAP50-95(B)'].idxmax() + 1
        best_map = df['metrics/mAP50-95(B)'].max()
        ax.plot(best_epoch, best_map, 'ro', markersize=12)
        ax.axhline(y=best_map, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_map:.4f}')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('mAP50-95', fontsize=11, fontweight='bold')
    ax.set_title('mAP50-95 Evolution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    # 3.3 Precision
    ax = axes[1, 0]
    if 'metrics/precision(B)' in df.columns:
        ax.plot(epochs, df['metrics/precision(B)'], linewidth=2.5, color='#3498db', label='Precision')
        best_epoch = df['metrics/precision(B)'].idxmax() + 1
        best_prec = df['metrics/precision(B)'].max()
        ax.plot(best_epoch, best_prec, 'ro', markersize=12)
        ax.axhline(y=best_prec, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_prec:.4f}')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Precision', fontsize=11, fontweight='bold')
    ax.set_title('Precision Evolution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    # 3.4 Recall
    ax = axes[1, 1]
    if 'metrics/recall(B)' in df.columns:
        ax.plot(epochs, df['metrics/recall(B)'], linewidth=2.5, color='#f39c12', label='Recall')
        best_epoch = df['metrics/recall(B)'].idxmax() + 1
        best_rec = df['metrics/recall(B)'].max()
        ax.plot(best_epoch, best_rec, 'ro', markersize=12)
        ax.axhline(y=best_rec, color='r', linestyle='--', alpha=0.5, label=f'Best: {best_rec:.4f}')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Recall', fontsize=11, fontweight='bold')
    ax.set_title('Recall Evolution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(graphs_dir / '03_metrics_evolution.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: metrics_evolution.png")

    # ========== 4. FINAL SPLIT EVALUATION (TRAIN/VAL/TEST) ==========
    print("4. Creating separate and combined evaluation graphs for Train, Val, Test...")
    try:
        if train_metrics and val_metrics and test_metrics:
            fig = plt.figure(figsize=(20, 12))
            fig.suptitle('Model Evaluation: Train vs Validation vs Test', fontsize=18, fontweight='bold')
            gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

            metrics_names = ['mAP50', 'mAP50-95', 'Precision', 'Recall']
            
            def get_vals(metrics):
                return [metrics.box.map50, metrics.box.map, metrics.box.mp, metrics.box.mr]

            train_vals = get_vals(train_metrics)
            val_vals = get_vals(val_metrics)
            test_vals = get_vals(test_metrics)

            colors_train = ['#2ecc71']*4
            colors_val = ['#3498db']*4
            colors_test = ['#e74c3c']*4

            # SEPARATE GRAPHS (Top row)
            # 4.1 Train
            ax1 = fig.add_subplot(gs[0, 0])
            bars1 = ax1.bar(metrics_names, train_vals, color=colors_train, alpha=0.8, edgecolor='black', linewidth=1.5)
            ax1.set_title('Training Set Metrics', fontsize=14, fontweight='bold')
            ax1.set_ylim([0, 1])
            ax1.set_ylabel('Score', fontsize=12, fontweight='bold')
            ax1.grid(True, alpha=0.3, axis='y')
            for bar, val in zip(bars1, train_vals):
                ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

            # 4.2 Val
            ax2 = fig.add_subplot(gs[0, 1])
            bars2 = ax2.bar(metrics_names, val_vals, color=colors_val, alpha=0.8, edgecolor='black', linewidth=1.5)
            ax2.set_title('Validation Set Metrics', fontsize=14, fontweight='bold')
            ax2.set_ylim([0, 1])
            ax2.grid(True, alpha=0.3, axis='y')
            for bar, val in zip(bars2, val_vals):
                ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

            # 4.3 Test
            ax3 = fig.add_subplot(gs[0, 2])
            bars3 = ax3.bar(metrics_names, test_vals, color=colors_test, alpha=0.8, edgecolor='black', linewidth=1.5)
            ax3.set_title('Test Set Metrics', fontsize=14, fontweight='bold')
            ax3.set_ylim([0, 1])
            ax3.grid(True, alpha=0.3, axis='y')
            for bar, val in zip(bars3, test_vals):
                ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height(), f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

            # COMBINED GRAPH (Bottom row, spans all columns)
            ax_comb = fig.add_subplot(gs[1, :])
            x = np.arange(len(metrics_names))
            width = 0.25

            ax_comb.bar(x - width, train_vals, width, label='Train', color='#2ecc71', alpha=0.8, edgecolor='black', linewidth=1.5)
            ax_comb.bar(x, val_vals, width, label='Validation', color='#3498db', alpha=0.8, edgecolor='black', linewidth=1.5)
            ax_comb.bar(x + width, test_vals, width, label='Test', color='#e74c3c', alpha=0.8, edgecolor='black', linewidth=1.5)

            ax_comb.set_ylabel('Score', fontsize=12, fontweight='bold')
            ax_comb.set_title('Combined Metrics Comparison', fontsize=14, fontweight='bold')
            ax_comb.set_xticks(x)
            ax_comb.set_xticklabels(metrics_names)
            ax_comb.legend(fontsize=12, loc='lower right')
            ax_comb.set_ylim([0, 1])
            ax_comb.grid(True, alpha=0.3, axis='y')

            # Add labels for combined graph
            for i in range(len(metrics_names)):
                ax_comb.text(i - width, train_vals[i], f'{train_vals[i]:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
                ax_comb.text(i, val_vals[i], f'{val_vals[i]:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
                ax_comb.text(i + width, test_vals[i], f'{test_vals[i]:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

            plt.tight_layout()
            plt.savefig(graphs_dir / '04_split_evaluation_separate_and_combine.png', dpi=300, bbox_inches='tight')
            plt.close()
            print(f"   ✓ Saved: 04_split_evaluation_separate_and_combine.png")

            # Save metrics to text file
            metrics_file = graphs_dir / 'evaluation_metrics_summary.txt'
            with open(metrics_file, 'w') as f:
                f.write("="*60 + "\n")
                f.write(f"{'Metric':<15} | {'Train':<12} | {'Val':<12} | {'Test':<12}\n")
                f.write("="*60 + "\n")
                for i, name in enumerate(metrics_names):
                    f.write(f"{name:<15} | {train_vals[i]:.6f}     | {val_vals[i]:.6f}     | {test_vals[i]:.6f}\n")
            print(f"   ✓ Saved: evaluation_metrics_summary.txt")

    except Exception as e:
        print(f"   ⚠ Warning: Could not create split evaluation graphs: {e}")

    # ========== 5. LEARNING RATE & OPTIMIZATION ==========
    print("5. Creating learning rate and optimization analysis...")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Learning Rate Schedule & Optimization', fontsize=16, fontweight='bold')

    # Learning rate
    ax = axes[0]
    if 'lr/pg0' in df.columns:
        ax.plot(epochs, df['lr/pg0'], linewidth=2.5, color='#9b59b6')
        ax.fill_between(epochs, df['lr/pg0'], alpha=0.3, color='#9b59b6')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Learning Rate', fontsize=11, fontweight='bold')
    ax.set_title('Learning Rate Schedule', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')

    # Loss vs Learning Rate (if available)
    ax = axes[1]
    if 'lr/pg0' in df.columns and 'val/box_loss' in df.columns:
        scatter = ax.scatter(df['lr/pg0'], df['val/box_loss'],
                           c=epochs, cmap='viridis', s=50, alpha=0.6, edgecolors='black')
        ax.set_xlabel('Learning Rate', fontsize=11, fontweight='bold')
        ax.set_ylabel('Validation Box Loss', fontsize=11, fontweight='bold')
        ax.set_title('Loss vs Learning Rate', fontsize=13, fontweight='bold')
        ax.set_xscale('log')
        ax.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax, label='Epoch')

    plt.tight_layout()
    plt.savefig(graphs_dir / '05_learning_rate_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: learning_rate_analysis.png")

    # ========== 6. FINAL SUMMARY REPORT ==========
    print("6. Creating final summary report...")
    fig = plt.figure(figsize=(16, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

    # Summary statistics
    ax_summary = fig.add_subplot(gs[:, :])
    ax_summary.axis('off')

    summary_text = []
    summary_text.append("TRAINING SUMMARY REPORT")
    summary_text.append("="*60)
    summary_text.append("")

    if 'metrics/mAP50(B)' in df.columns:
        summary_text.append(f"Best mAP50:           {df['metrics/mAP50(B)'].max():.6f} (Epoch {df['metrics/mAP50(B)'].idxmax()+1})")
        summary_text.append(f"Final mAP50:          {df['metrics/mAP50(B)'].iloc[-1]:.6f}")
    if 'metrics/mAP50-95(B)' in df.columns:
        summary_text.append(f"Best mAP50-95:         {df['metrics/mAP50-95(B)'].max():.6f} (Epoch {df['metrics/mAP50-95(B)'].idxmax()+1})")
        summary_text.append(f"Final mAP50-95:       {df['metrics/mAP50-95(B)'].iloc[-1]:.6f}")
    if 'metrics/precision(B)' in df.columns:
        summary_text.append(f"Best Precision:       {df['metrics/precision(B)'].max():.6f} (Epoch {df['metrics/precision(B)'].idxmax()+1})")
        summary_text.append(f"Final Precision:      {df['metrics/precision(B)'].iloc[-1]:.6f}")
    if 'metrics/recall(B)' in df.columns:
        summary_text.append(f"Best Recall:          {df['metrics/recall(B)'].max():.6f} (Epoch {df['metrics/recall(B)'].idxmax()+1})")
        summary_text.append(f"Final Recall:         {df['metrics/recall(B)'].iloc[-1]:.6f}")

    summary_text.append("")
    summary_text.append("="*60)
    summary_text.append(f"Total Epochs:          {len(df)}")
    if 'val/box_loss' in df.columns:
        summary_text.append(f"Best Validation Loss: {df['val/box_loss'].min():.6f} (Epoch {df['val/box_loss'].idxmin()+1})")
    summary_text.append("")
    summary_text.append("All graphs saved in: comprehensive_graphs/")

    ax_summary.text(0.5, 0.5, '\n'.join(summary_text),
                   fontsize=12, fontfamily='monospace',
                   verticalalignment='center', horizontalalignment='center',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                   transform=ax_summary.transAxes)

    plt.savefig(graphs_dir / '06_summary_report.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: summary_report.png")

    print("\n" + "="*50)
    print("All comprehensive graphs generated successfully!")
    print(f"Graphs saved in: {graphs_dir}")
    print("="*50)

def main():
    # Set up paths
    dataset_yaml = Path("/kaggle/working/dataset.yaml")
    project_dir = Path("/kaggle/working/runs")

    # Check if dataset config exists
    if not dataset_yaml.exists():
        print(f"Error: dataset.yaml not found at {dataset_yaml}")
        return

    # Initialize YOLO model
    # Using YOLOv8n (nano) for lightweight model, can change to:
    # - YOLOv8n: nano (fastest, smallest)
    # - YOLOv8s: small
    # - YOLOv8m: medium
    # - YOLOv8l: large
    # - YOLOv8x: extra large (most accurate)

    print("Initializing YOLOv8x model for high accuracy UAV detection...")
    model = YOLO('yolo11x.pt')  # Start with the latest Ultralytics extra-large model

    # Auto-detect device (GPU or CPU)
    import torch
    if torch.cuda.is_available():
        gpu_count = torch.cuda.device_count()
        if gpu_count > 1:
            device = [i for i in range(gpu_count)]  # Use all available GPUs
            device_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
            print(f"\n✓ GPUs detected: {device_names}")
            print(f"  Multiple GPUs detected! Using {gpu_count} GPUs for distributed training.")
        else:
            device = 0  # Use first GPU
        if gpu_count == 1:
                    device_name = torch.cuda.get_device_name(0)
        if gpu_count == 1:
                    print(f"\n✓ GPU detected: {device_name}")
        print(f"  CUDA Version: {torch.version.cuda}")
        print(f"  GPU will be used for training")
    else:
        device = 'cpu'
        print(f"\n⚠ GPU not available (PyTorch CPU-only version installed)")
        print(f"  Training will use CPU (slower but will work)")
        print(f"  To use GPU, install PyTorch with CUDA support:")
        print(f"    Visit: https://pytorch.org/get-started/locally/")

    # Training parameters
    print("\nStarting training...")
    print(f"Dataset configuration: {dataset_yaml}")

    # Adjust batch size based on device
    batch_size = 16 if device != 'cpu' else 4  # Smaller batch for CPU

    results = model.train(
        data=str(dataset_yaml),           # Path to dataset yaml file
        epochs=30,                        # Number of training epochs
        imgsz=640,                         # Image size (640x640)
        batch=batch_size,                  # Batch size (adjusted based on device)
        device=device,                     # Auto-detected device (GPU or CPU)
        workers=8,                         # Number of worker threads
        project=str(project_dir),          # Project directory
        name="uav_detection",              # Experiment name
        exist_ok=True,                     # Overwrite existing experiment
        pretrained=True,                   # Use pretrained weights
        optimizer='AdamW',                 # Optimizer
        verbose=True,                      # Verbose output
        seed=42,                           # Random seed for reproducibility
        deterministic=True,                # Deterministic training
        single_cls=True,                   # Single class mode (drone detection only)
        rect=False,                        # Rectangular training
        cos_lr=False,                      # Cosine learning rate scheduler
        close_mosaic=10,                   # Disable mosaic augmentation for last N epochs
        resume=False,                      # Resume training from last checkpoint
        amp=(device != 'cpu'),            # Automatic Mixed Precision (only for GPU)
        fraction=1.0,                      # Dataset fraction to use
        profile=False,                     # Profile ONNX and TensorRT speeds
        freeze=None,                       # Freeze layers (backbone=10, freeze3=3)
        lr0=0.01,                          # Initial learning rate
        lrf=0.01,                          # Final learning rate (lr0 * lrf)
        momentum=0.937,                    # SGD momentum/Adam beta1
        weight_decay=0.0005,               # Optimizer weight decay
        warmup_epochs=3.0,                 # Warmup epochs
        warmup_momentum=0.8,               # Warmup initial momentum
        warmup_bias_lr=0.1,                # Warmup initial bias lr
        box=7.5,                           # Box loss gain
        cls=0.5,                           # Class loss gain
        dfl=1.5,                           # DFL loss gain
        pose=12.0,                         # Pose loss gain
        kobj=1.0,                          # Keypoint obj loss gain
        label_smoothing=0.0,               # Label smoothing
        nbs=64,                            # Nominal batch size
        overlap_mask=True,                 # Masks should overlap during training
        mask_ratio=4,                      # Mask downsample ratio
        dropout=0.0,                       # Use dropout regularization
        val=True,                          # Validate/test during training
    )

    print("\n" + "="*50)
    print("Training completed!")
    print("="*50)

    # Print results summary
    print(f"\nBest model saved at: {model.trainer.save_dir}/weights/best.pt")
    print(f"Last model saved at: {model.trainer.save_dir}/weights/last.pt")

    # Load best model for evaluation
    best_model_path = Path(model.trainer.save_dir) / "weights" / "best.pt"
    if best_model_path.exists():
        print("\nLoading best model for comprehensive evaluation...")
        eval_model = YOLO(str(best_model_path))
    else:
        eval_model = model

    # Validate the model on all splits
    print("\n" + "="*50)
    print("Evaluating model on all dataset splits (Train, Val, Test)...")
    print("="*50)
    
    # Validation on Validation set
    print("\n1. Running evaluation on Validation set...")
    val_metrics = eval_model.val(data=str(dataset_yaml), split='val', device=device)
    print(f"Validation mAP50: {val_metrics.box.map50:.4f}")

    # Validation on Train set
    print("\n2. Running evaluation on Training set...")
    train_metrics = eval_model.val(data=str(dataset_yaml), split='train', device=device)
    print(f"Training mAP50: {train_metrics.box.map50:.4f}")

    # Validation on Test set
    print("\n3. Running evaluation on Test set...")
    test_metrics = eval_model.val(data=str(dataset_yaml), split='test', device=device)
    print(f"Test mAP50: {test_metrics.box.map50:.4f}")

    # Generate comprehensive graphs
    create_comprehensive_graphs(model.trainer.save_dir, eval_model, dataset_yaml, train_metrics, val_metrics, test_metrics)

    # Export model to different formats
    print("\nExporting model to different formats...")
    model.export(format='onnx', device=device)  # Export to ONNX
    model.export(format='torchscript', device=device)  # Export to TorchScript

    print("\nModel export completed!")
    print(f"Exported models saved in: {model.trainer.save_dir}/weights/")
    print(f"\nAll comprehensive graphs saved in: {model.trainer.save_dir}/comprehensive_graphs/")

if __name__ == "__main__":
    main()

In [ ]:
import zipfile
import os

output_zip_path = '/content/runs.zip'
directory_to_zip = '/content/runs'

try:
    with zipfile.ZipFile(output_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(directory_to_zip):
            for file in files:
                file_path = os.path.join(root, file)
                # Arcname is the path inside the zip file
                arcname = os.path.relpath(file_path, directory_to_zip)
                zipf.write(file_path, arcname)
    print(f"Successfully created zip archive: {output_zip_path}")
    print(f"You can now download '{output_zip_path}' from the Colab file browser.")
except Exception as e:
    print(f"An error occurred during zipping: {e}")
